## Radiomics extraction - CRLM example


* Héctor Henríquez Leighton MD, MS

In [ ]:
!pip install SimpleITK
!pip install lifelines
!pip install pyradiomics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from scipy.ndimage import median_filter
from scipy.ndimage import binary_erosion
from skimage import measure
from skimage.measure import label, regionprops
import os
from lifelines import KaplanMeierFitter
import nibabel as nib
import SimpleITK as sitk
import shutil
import radiomics
from radiomics import featureextractor

%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [ ]:
def show_slice_window(slice, level, window):
    """
    input: imagen array 2D,
    permite ajustar ventana y nivel para mejorar contraste de la imagen.
    output: imagen array 2D ventaneada.
   """
    max = level + window/2
    min = level - window/2
    slice = slice.clip(min,max)
    return(slice)

In [ ]:

path_img = '/content/ct_volume_image_CRLM-CT-1001.nii.gz'
img_sitk = sitk.ReadImage(path_img)
img_array = sitk.GetArrayFromImage(img_sitk)

path_mask = '/content/combined_mask_seg_CRLM-CT-1001.nii.gz'
mask_sitk = sitk.ReadImage(path_mask)
mask_array = sitk.GetArrayFromImage(mask_sitk)

print(img_array.shape)
print(mask_array.shape)

In [ ]:
## Leer máscaras de un ejemplo
axial = 100

plt.figure()
plt.imshow(show_slice_window(img_array[axial],100,300), cmap='gray')
plt.imshow(mask_array[axial], cmap='jet', alpha=0.5)
plt.axis('off')
plt.show()

In [ ]:
## Lectura de máscara de remanente hepático:

axial = 100

mask_lrem = np.where(mask_array == 2,1,0)


plt.figure()
plt.imshow(show_slice_window(img_array[axial],100,300), cmap='gray')
plt.imshow(mask_lrem[axial], cmap='jet', alpha=0.5)
plt.axis('off')
plt.show()

## Definición de parámetros para extracción sin filtros

In [ ]:
# definición de parámetros
params = {
    'imageType': {
        'Original': {},
    },
    'setting': {
        'normalize': True,
        'binWidth': 5,
        'resampledPixelSpacing': [1, 1, 1],
        'interpolator': 'sitkBSpline',
        'preCrop': True,
        'padDistance': 5,
        'label': 2
    },
    'featureClass': {
        'shape': [],
        'firstorder': [],
        'glcm': [],
        'glrlm': [],
        'glszm': [],
        'gldm': [],
        'ngtdm': []
    }
}

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor(params)

result = extractor.execute(img_sitk, mask_sitk)

In [ ]:
tabla = pd.DataFrame([result])
tabla.shape

In [ ]:
tabla

## Definición de parámetros para extracción CON filtros

In [ ]:
# definición de parámetros
params_filters = {
    'imageType': {
        'Original': {},
        'LoG': {'sigma': [1.0, 2.0, 3.0]},
        'Wavelet': {},
        'Square': {},
        'SquareRoot': {},
        'Logarithm': {},
        'Exponential': {},
        'Gradient': {}
    },
    'setting': {
        'normalize': True,
        'binWidth': 5,
        'resampledPixelSpacing': [1, 1, 1],
        'interpolator': 'sitkBSpline',
        'preCrop': True,
        'padDistance': 5,
        'label': 2
    },
    'featureClass': {
        'shape': [],
        'firstorder': [],
        'glcm': [],
        'glrlm': [],
        'glszm': [],
        'gldm': [],
        'ngtdm': []
    }
}

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor(params_filters)

result2 = extractor.execute(img_sitk, mask_sitk)

In [ ]:
tabla_con_filtros = pd.DataFrame([result2])
tabla_con_filtros.shape

In [ ]:
tabla_con_filtros